# HESE Event Comparison

1. Load all events from the per-year EvtGen.h5 files.
2. Compare run coverage against the L5 i3 files from eyildizci's processing.
3. Compare run coverage against the HESE12 i3 files.
4. Apply `reco_energy > 60 TeV` cut (stored separately as `df_evtgen_cut`).
5. Load reference events from HESE12.hdf5.
6. Verify the topology sub-files add up to HESE12.hdf5.
7. Compare EvtGen (after cut) against HESE12.

In [1]:
import re
import tables
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Collect events from per-year EvtGen files

In [2]:
BASE_PATH = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/v1"

DATASETS = [
    "IC79_2010",
    "IC86_2011",
    "IC86_2012",
    "IC86_2013",
    "IC86_2014",
    "IC86_2015",
    "IC86_2016",
    "IC86_2017",
    "IC86_2018",
    "IC86_2019",
    "IC86_2020",
    "IC86_2021",
]

In [6]:
records = []

for dataset in DATASETS:
    path = f"{BASE_PATH}/{dataset}/EvtGen/EvtGen.h5"
    with tables.open_file(path, "r") as f:
        df_hdr = pd.DataFrame({
            "run":   f.root.I3EventHeader.col("Run"),
            "event": f.root.I3EventHeader.col("Event"),
        })
        df_en = pd.DataFrame({
            "run":         f.root.RecoETot.col("Run"),
            "event":       f.root.RecoETot.col("Event"),
            "reco_energy": f.root.RecoETot.col("value"),
        })
    df = df_hdr.merge(df_en, on=["run", "event"], how="left")
    df.insert(0, "dataset", dataset)
    records.append(df)
    print(f"{dataset}: {len(df)} events")

df_evtgen = pd.concat(records, ignore_index=True)
print(f"\nTotal events: {len(df_evtgen)}")
df_evtgen[ df_evtgen["dataset"] == "IC86_2011" ]

IC79_2010: 7 events
IC86_2011: 19 events
IC86_2012: 8 events
IC86_2013: 13 events
IC86_2014: 15 events
IC86_2015: 8 events
IC86_2016: 22 events
IC86_2017: 18 events
IC86_2018: 16 events
IC86_2019: 19 events
IC86_2020: 9 events
IC86_2021: 12 events

Total events: 166


,dataset,run,event,reco_energy
7,IC86_2011,118178,66452255,9.007058e+04
8,IC86_2011,118283,9445773,8.299118e+04
9,IC86_2011,118381,19162840,8.938133e+04
10,IC86_2011,118435,58198553,2.766010e+05
11,IC86_2011,118545,63733662,9.648540e+05
12,IC86_2011,118549,11722208,5.100263e+04
13,IC86_2011,118602,23096391,3.222959e+04
14,IC86_2011,118607,40435683,1.844783e+05
15,IC86_2011,119196,37713300,3.639132e+04
16,IC86_2011,119214,8606380,6.532968e+04


## 2. Compare EvtGen run coverage with L5 processing (eyildizci)

In [11]:
L5_BASE = "/data/user/eyildizci/hese_tracks_processing/L5"
L5_YEARS = [str(y) for y in range(2011, 2023)]

l5_records = []
for year in L5_YEARS:
    folder = Path(L5_BASE) / f"new_IC86_{year}"
    for fname in sorted(folder.iterdir()):
        m = re.search(r"Run(\d+)", fname.name)
        if m:
            l5_records.append({"dataset": f"IC86_{year}", "run": int(m.group(1))})

df_l5 = pd.DataFrame(l5_records)
print(f"L5 files (runs): {len(df_l5)}")
df_l5.groupby("dataset").size().rename("n_runs")

L5 files (runs): 179


dataset
IC86_2011    19
IC86_2012     8
IC86_2013    15
IC86_2014    15
IC86_2015     8
IC86_2016    22
IC86_2017    18
IC86_2018    16
IC86_2019    18
IC86_2020     9
IC86_2021    12
IC86_2022    19
Name: n_runs, dtype: int64

In [12]:
# Compare at run level: EvtGen unique runs vs L5 runs
evtgen_runs = set(zip(df_evtgen["dataset"], df_evtgen["run"]))
l5_runs     = set(zip(df_l5["dataset"],     df_l5["run"]))

only_evtgen = evtgen_runs - l5_runs
only_l5     = l5_runs - evtgen_runs

print(f"Runs in EvtGen also in L5:      {len(evtgen_runs - only_evtgen)} / {len(evtgen_runs)}")
print(f"Runs in EvtGen but NOT in L5:   {len(only_evtgen)}")
print(f"Runs in L5 but NOT in EvtGen:   {len(only_l5)}")

Runs in EvtGen also in L5:      157 / 164
Runs in EvtGen but NOT in L5:   7
Runs in L5 but NOT in EvtGen:   22


In [13]:
# EvtGen events whose run is not in L5
df_evtgen["run_in_l5"] = [
    (ds, r) in l5_runs
    for ds, r in zip(df_evtgen["dataset"], df_evtgen["run"])
]

df_not_in_l5 = (
    df_evtgen[~df_evtgen["run_in_l5"]]
    [["dataset", "run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"EvtGen events whose run is not in L5: {len(df_not_in_l5)}")
df_not_in_l5

EvtGen events whose run is not in L5: 8


,dataset,run,event,reco_energy
0,IC79_2010,115994,2538090,44914.090653
1,IC79_2010,115994,29874216,102341.693610
2,IC79_2010,116528,52433389,68520.107392
3,IC79_2010,116698,10198436,148738.586031
4,IC79_2010,117371,31623515,31055.609948
5,IC79_2010,117782,49441871,23559.228620
6,IC79_2010,118145,5142726,58627.611326
7,IC86_2014,125826,470241,416499.030804


In [14]:
# L5 runs not represented in EvtGen
df_l5["in_evtgen"] = [(ds, r) in evtgen_runs for ds, r in zip(df_l5["dataset"], df_l5["run"])]

df_l5_missing = df_l5[~df_l5["in_evtgen"]].reset_index(drop=True)
print(f"L5 runs not in EvtGen: {len(df_l5_missing)}")
df_l5_missing

L5 runs not in EvtGen: 22


,dataset,run,in_evtgen
0,IC86_2013,122604,False
1,IC86_2013,122752,False
2,IC86_2014,125627,False
3,IC86_2022,136918,False
4,IC86_2022,136985,False
5,IC86_2022,137007,False
6,IC86_2022,137160,False
7,IC86_2022,137167,False
8,IC86_2022,137489,False
9,IC86_2022,137512,False


## 3. Compare EvtGen run coverage with HESE12 i3 files

In [15]:
HESE12_I3_BASE = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr"

i3_records = []
for dataset in DATASETS:
    folder = Path(HESE12_I3_BASE) / dataset
    seen_runs = set()
    for fname in sorted(folder.iterdir()):
        m = re.search(r"Run(\d+)", fname.name)
        if m:
            run = int(m.group(1))
            if run not in seen_runs:
                seen_runs.add(run)
                i3_records.append({"dataset": dataset, "run": run})

df_hese12_i3 = pd.DataFrame(i3_records)
print(f"HESE12 i3 unique runs: {len(df_hese12_i3)}")
df_hese12_i3.groupby("dataset").size().rename("n_runs")

HESE12 i3 unique runs: 163


dataset
IC79_2010     6
IC86_2011    19
IC86_2012     8
IC86_2013    15
IC86_2014    14
IC86_2015     8
IC86_2016    22
IC86_2017    18
IC86_2018    14
IC86_2019    18
IC86_2020     9
IC86_2021    12
Name: n_runs, dtype: int64

In [16]:
# Compare at run level: EvtGen unique runs vs HESE12 i3 runs
hese12_i3_runs = set(zip(df_hese12_i3["dataset"], df_hese12_i3["run"]))

only_evtgen_i3 = evtgen_runs - hese12_i3_runs
only_i3        = hese12_i3_runs - evtgen_runs

print(f"Runs in EvtGen also in HESE12 i3:    {len(evtgen_runs - only_evtgen_i3)} / {len(evtgen_runs)}")
print(f"Runs in EvtGen but NOT in HESE12 i3: {len(only_evtgen_i3)}")
print(f"Runs in HESE12 i3 but NOT in EvtGen: {len(only_i3)}")

# EvtGen events whose run is not in the HESE12 i3 files
df_evtgen["run_in_hese12_i3"] = [
    (ds, r) in hese12_i3_runs
    for ds, r in zip(df_evtgen["dataset"], df_evtgen["run"])
]

df_not_in_hese12_i3 = (
    df_evtgen[~df_evtgen["run_in_hese12_i3"]]
    [["dataset", "run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"\nEvtGen events whose run is not in HESE12 i3 files: {len(df_not_in_hese12_i3)}")
display(df_not_in_hese12_i3)

# HESE12 i3 runs not represented in EvtGen
df_hese12_i3["in_evtgen"] = [(ds, r) in evtgen_runs for ds, r in zip(df_hese12_i3["dataset"], df_hese12_i3["run"])]
df_i3_missing = df_hese12_i3[~df_hese12_i3["in_evtgen"]].reset_index(drop=True)
print(f"\nHESE12 i3 runs not in EvtGen: {len(df_i3_missing)}")
display(df_i3_missing)

Runs in EvtGen also in HESE12 i3:    161 / 164
Runs in EvtGen but NOT in HESE12 i3: 3
Runs in HESE12 i3 but NOT in EvtGen: 2

EvtGen events whose run is not in HESE12 i3 files: 3


,dataset,run,event,reco_energy
0,IC86_2014,125826,470241,416499.030804
1,IC86_2018,131680,66412090,225256.253680
2,IC86_2018,132143,36142391,199159.361695



HESE12 i3 runs not in EvtGen: 2


,dataset,run,in_evtgen
0,IC86_2013,122604,False
1,IC86_2013,122752,False


## 4. Apply reco_energy > 60 TeV cut

In [17]:
ENERGY_CUT = 60e3  # GeV

df_evtgen_cut = df_evtgen[df_evtgen["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"Events before cut: {len(df_evtgen)}")
print(f"Events after cut:  {len(df_evtgen_cut)}")
df_evtgen_cut.groupby("dataset").size().rename("n_events")

Events before cut: 166
Events after cut:  97


dataset
IC79_2010     3
IC86_2011    11
IC86_2012     3
IC86_2013     9
IC86_2014    12
IC86_2015     6
IC86_2016    13
IC86_2017    12
IC86_2018     7
IC86_2019     8
IC86_2020     6
IC86_2021     7
Name: n_events, dtype: int64

## 5. Load reference events from HESE12.hdf5

In [18]:
HESE12_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/HESE12.hdf5"

with tables.open_file(HESE12_PATH, "r") as f:
    df_hdr_ref = pd.DataFrame({
        "run":   f.root.I3EventHeader.col("Run"),
        "event": f.root.I3EventHeader.col("Event"),
    })
    df_en_ref = pd.DataFrame({
        "run":         f.root.RecoETot.col("Run"),
        "event":       f.root.RecoETot.col("Event"),
        "reco_energy": f.root.RecoETot.col("value"),
    })

df_hese12 = df_hdr_ref.merge(df_en_ref, on=["run", "event"], how="left")
print(f"HESE12 events: {len(df_hese12)}")
df_hese12.head()

HESE12 events: 97


,run,event,reco_energy
0,134912,80902176,106029.802180
1,131502,58093123,401730.596192
2,131505,52347097,150473.232499
3,129253,4711217,259569.685977
4,128619,66564632,151213.222561


## 6. Check topology sub-files add up to HESE12.hdf5

In [19]:
BFR_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr"

TOPOLOGY_FILES = {
    "Cascades":       f"{BFR_PATH}/HESE12_Cascades.hdf5",
    "DoubleCascades": f"{BFR_PATH}/HESE12_DoubleCascades.hdf5",
    "Tracks":         f"{BFR_PATH}/HESE12_Tracks.hdf5",
}

topo_dfs = {}
for topology, path in TOPOLOGY_FILES.items():
    with tables.open_file(path, "r") as f:
        df_hdr = pd.DataFrame({
            "run":   f.root.I3EventHeader.col("Run"),
            "event": f.root.I3EventHeader.col("Event"),
        })
        df_en = pd.DataFrame({
            "run":         f.root.RecoETot.col("Run"),
            "event":       f.root.RecoETot.col("Event"),
            "reco_energy": f.root.RecoETot.col("value"),
        })
    df = df_hdr.merge(df_en, on=["run", "event"], how="left")
    df.insert(0, "topology", topology)
    topo_dfs[topology] = df
    print(f"{topology}: {len(df)} events")

df_topo = pd.concat(topo_dfs.values(), ignore_index=True)
print(f"\nTotal across topology files: {len(df_topo)}")
print(f"Total in HESE12.hdf5:        {len(df_hese12)}")

Cascades: 64 events
DoubleCascades: 5 events
Tracks: 28 events

Total across topology files: 97
Total in HESE12.hdf5:        97


In [20]:
topo_set   = set(zip(df_topo["run"],   df_topo["event"]))
hese12_set = set(zip(df_hese12["run"], df_hese12["event"]))

only_in_topo   = topo_set - hese12_set
only_in_hese12 = hese12_set - topo_set

print(f"Events in topology files but not in HESE12.hdf5: {len(only_in_topo)}")
print(f"Events in HESE12.hdf5 but not in topology files: {len(only_in_hese12)}")

if not only_in_topo and not only_in_hese12:
    print("\nTopology files add up exactly to HESE12.hdf5.")
else:
    if only_in_topo:
        extra = df_topo[df_topo.apply(lambda r: (r.run, r.event) in only_in_topo, axis=1)]
        print("\nExtra events in topology files:")
        display(extra[["topology", "run", "event", "reco_energy"]].reset_index(drop=True))
    if only_in_hese12:
        missing = df_hese12[df_hese12.apply(lambda r: (r.run, r.event) in only_in_hese12, axis=1)]
        print("\nEvents missing from topology files:")
        display(missing[["run", "event", "reco_energy"]].reset_index(drop=True))

Events in topology files but not in HESE12.hdf5: 0
Events in HESE12.hdf5 but not in topology files: 0

Topology files add up exactly to HESE12.hdf5.


## 7. Compare EvtGen (after cut) with HESE12

In [21]:
hese12_set = set(zip(df_hese12["run"], df_hese12["event"]))

df_evtgen_cut["in_hese12"] = [
    (r, e) in hese12_set
    for r, e in zip(df_evtgen_cut["run"], df_evtgen_cut["event"])
]

n_matched = df_evtgen_cut["in_hese12"].sum()
print(f"EvtGen events (after cut) also in HESE12: {n_matched} / {len(df_evtgen_cut)}")

EvtGen events (after cut) also in HESE12: 95 / 97


In [22]:
# Summary per dataset
summary = df_evtgen_cut.groupby("dataset")["in_hese12"].agg(
    n_events="count", n_in_hese12="sum"
)
summary["fraction"] = summary["n_in_hese12"] / summary["n_events"]
print(summary.to_string())

           n_events  n_in_hese12  fraction
dataset                                   
IC79_2010         3            3  1.000000
IC86_2011        11           11  1.000000
IC86_2012         3            3  1.000000
IC86_2013         9            9  1.000000
IC86_2014        12           11  0.916667
IC86_2015         6            6  1.000000
IC86_2016        13           12  0.923077
IC86_2017        12           12  1.000000
IC86_2018         7            7  1.000000
IC86_2019         8            8  1.000000
IC86_2020         6            6  1.000000
IC86_2021         7            7  1.000000


In [23]:
# Events in EvtGen (after cut) but NOT in HESE12
df_not_in_hese12 = (
    df_evtgen_cut[~df_evtgen_cut["in_hese12"]]
    [["dataset", "run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"Events in EvtGen (after cut) but missing from HESE12: {len(df_not_in_hese12)}")
df_not_in_hese12

Events in EvtGen (after cut) but missing from HESE12: 2


,dataset,run,event,reco_energy
0,IC86_2014,125826,470241,416499.030804
1,IC86_2016,128672,38561326,83136.722946


In [24]:
# Events in HESE12 but NOT in any EvtGen file (after cut)
evtgen_cut_set = set(zip(df_evtgen_cut["run"], df_evtgen_cut["event"]))
df_hese12["in_evtgen"] = [
    (r, e) in evtgen_cut_set
    for r, e in zip(df_hese12["run"], df_hese12["event"])
]

df_not_in_evtgen = (
    df_hese12[~df_hese12["in_evtgen"]]
    [["run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"HESE12 events not found in EvtGen (after cut): {len(df_not_in_evtgen)}")
df_not_in_evtgen

HESE12 events not found in EvtGen (after cut): 2


,run,event,reco_energy
0,122752,41309299,156888.505974
1,127210,51027948,60806.979712
